<a href="https://colab.research.google.com/github/pachterlab/varseek-examples/blob/main/vk_denovo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [vk denovo](https://github.com/pachterlab/varseek) demonstration
Call de novo variants from scRNA-seq reads, build a varseek reference from those variants, and count variant-supporting reads with vk count. This notebook uses a [10x PBMC 1k dataset](https://www.10xgenomics.com/datasets/1-k-pbm-cs-from-a-healthy-donor-v-3-chemistry-3-standard-3-0-0) as an example.

Written by Joseph Rich.
___


### Install varseek, and import all packages

In [1]:
try:
    import varseek as vk
except ImportError:
    print("varseek not found, installing...")
    !pip install -U -q varseek

In [2]:
import os
import re
import anndata as ad
import pandas as pd
import pysam

import varseek as vk

In [3]:
# !pip install -q ipython-autotime
%load_ext autotime

time: 62.3 μs (started: 2026-06-15 08:45:35 -07:00)


### Define important paths

In [4]:
# input files
fastqs_dir = os.path.join("data", "pbmc_1k_v3_fastqs")
technology = "10xv3"
reference_dir = os.path.join("data", "reference")
sequences = os.path.join(reference_dir, "Homo_sapiens.GRCh38.dna.primary_assembly.fa")
gtf = os.path.join(reference_dir, "Homo_sapiens.GRCh38.114.gtf")

# vk denovo out
variants_dir = os.path.join("data", "pbmc_1k_v3_variants")
variants_vcf = os.path.join(variants_dir, "variants.vcf.gz")
variants = os.path.join(variants_dir, "variants.tsv")
denovo_bam_dir = os.path.join(variants_dir, "bams")
star_genome_index_dir = os.path.join(reference_dir, "star_index")
denovo_star_alignment_dir = os.path.join(variants_dir, "star_alignments")

# vk ref out
vk_ref_out_dir = os.path.join("data", "varseek_ref_out_pbmc_1k_v3")
vcrs_index = os.path.join(vk_ref_out_dir, "vcrs_index_denovo.idx")
vcrs_t2g = os.path.join(vk_ref_out_dir, "vcrs_t2g_denovo.txt")

# vk count out
vk_count_out_dir = os.path.join("data", "varseek_count_out_pbmc_1k_v3")
vk_count_variants_vcf = os.path.join(vk_count_out_dir, "varseek_variants.vcf")

# general parameters
w = 37
k = 41
min_counts_denovo = 3
threads = 16
min_counts_clean = 1

time: 987 μs (started: 2026-06-15 08:45:35 -07:00)


In [5]:
def adata_to_vcf(adata, vcf_out):
    if os.path.exists(vcf_out):
        print(f"{vcf_out} already exists, skipping VCF writing.")
        return

    if isinstance(adata, str):
        adata = ad.read_h5ad(adata)
    if not isinstance(adata, ad.AnnData):
        raise ValueError("adata must be either a file path or an AnnData object.")

    adata = adata[:, adata.X.sum(axis=0) > 0]  # filter to only keep variants present in at least 1 cell
    variants = adata.var_names.tolist()

    # Reference genome FASTA
    fasta = pysam.FastaFile(sequences)

    chromosomes = {str(i) for i in range(1, 23)}.union({"X", "Y", "MT"})

    # Flatten grouped variants (joined by ";") into one Series, preserving the
    # original order so the written VCF matches the old per-variant loop.
    var_series = pd.Series(variants, dtype="object").str.split(";").explode()
    var_series = var_series[var_series.str.len() > 0].reset_index(drop=True)

    # Vectorized HGVS parsing: every variant is exactly one of three forms.
    # str.extract returns NaN for every group on a non-matching row, so a
    # notna() check on a required group acts as the per-type mask.
    snv = var_series.str.extract(r"^(?P<CHROM>.+):g\.(?P<POS>\d+)(?P<REF>[ACGT]+)>(?P<ALT>[ACGT]+)$")
    ins = var_series.str.extract(r"^(?P<CHROM>.+):g\.(?P<POS>\d+)_(?P<END>\d+)ins(?P<INS>[ACGT]+)$")
    dele = var_series.str.extract(r"^(?P<CHROM>.+):g\.(?P<START>\d+)(?:_(?P<END>\d+))?del(?P<DEL>[ACGT]*)$")

    snv_mask = snv["CHROM"].notna()
    ins_mask = ins["CHROM"].notna()
    del_mask = dele["START"].notna()

    for var in var_series[~(snv_mask | ins_mask | del_mask)]:
        print(f"Could not parse: {var}")

    frames = []

    # --- Substitutions (no reference lookup needed) ---
    snv = snv[snv_mask & snv["CHROM"].isin(chromosomes)]
    if not snv.empty:
        frames.append(pd.DataFrame({
            "CHROM": snv["CHROM"],
            "POS": snv["POS"].astype(int),
            "REF": snv["REF"],
            "ALT": snv["ALT"],
        }, index=snv.index))

    # --- Insertions: prepend the left anchor base from the reference ---
    ins = ins[ins_mask & ins["CHROM"].isin(chromosomes)]
    if not ins.empty:
        ins_pos = ins["POS"].astype(int)
        anchor = pd.Series(
            [fasta.fetch(c, p - 1, p) for c, p in zip(ins["CHROM"], ins_pos)],
            index=ins.index,
        )
        frames.append(pd.DataFrame({
            "CHROM": ins["CHROM"],
            "POS": ins_pos,
            "REF": anchor,
            "ALT": anchor + ins["INS"],
        }, index=ins.index))

    # --- Deletions: anchor base before the deletion; fetch the deleted bases
    #     from the reference when HGVS omits them (e.g. "g.123del") ---
    dele = dele[del_mask & dele["CHROM"].isin(chromosomes)]
    if not dele.empty:
        start = dele["START"].astype(int)
        end = pd.to_numeric(dele["END"]).fillna(start).astype(int)
        anchor = pd.Series(
            [fasta.fetch(c, s - 2, s - 1) for c, s in zip(dele["CHROM"], start)],
            index=dele.index,
        )
        deleted = pd.Series(
            [d if d != "" else fasta.fetch(c, s - 1, e)
            for c, s, e, d in zip(dele["CHROM"], start, end, dele["DEL"])],
            index=dele.index,
        )
        frames.append(pd.DataFrame({
            "CHROM": dele["CHROM"],
            "POS": start - 1,
            "REF": anchor + deleted,
            "ALT": anchor,
        }, index=dele.index))

    # Concatenate all variant types and restore the original variant order.
    vcf_df = (pd.concat(frames).sort_index()
            if frames else pd.DataFrame(columns=["CHROM", "POS", "REF", "ALT"]))

    # Build every data line with vectorized string ops, then write in one shot.
    lines = (
        vcf_df["CHROM"].astype(str) + "\t"
        + vcf_df["POS"].astype(str) + "\t.\t"
        + vcf_df["REF"] + "\t"
        + vcf_df["ALT"] + "\t.\t.\t.\n"
    )

    with open(vcf_out, "w") as f:
        f.write("##fileformat=VCFv4.2\n")
        f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        f.write("".join(lines))

    print(f"Wrote {len(vcf_df):,} variants to {vcf_out}")

time: 2.31 ms (started: 2026-06-15 08:45:35 -07:00)


In [6]:
import os
import gzip
import subprocess

def is_vcf_normalized(vcf_path):
    """
    Returns True if the VCF has a header line indicating bcftools norm was run.
    """
    # Auto-detect if compressed
    open_func = gzip.open if vcf_path.endswith(".gz") else open

    with open_func(vcf_path, "rt") as f:
        for line in f:
            if line.startswith("##bcftools_normCommand="):
                return True
            # Headers always start with ##
            if not line.startswith("##"):
                # Stop at the column header line
                break
    return False

def ensure_genotype_column(vcf_path, sample_name="SAMPLE", genotype="1/1"):
    """hap.py requires at least one sample carrying genotypes. varseek emits a
    sites-only VCF (CHROM..INFO only), so inject a placeholder FORMAT/GT column
    if none is present. No-op for VCFs that already have sample columns."""
    open_func = gzip.open if vcf_path.endswith(".gz") else open

    with open_func(vcf_path, "rt") as f:
        lines = f.readlines()

    for line in lines:
        if line.startswith("#CHROM"):
            if len(line.rstrip("\n").split("\t")) > 8:
                return  # FORMAT + sample columns already present
            break

    gt_header = '##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">\n'
    new_lines = []
    inserted_header = False
    for line in lines:
        if line.startswith("##"):
            new_lines.append(line)
        elif line.startswith("#CHROM"):
            if not inserted_header:
                new_lines.append(gt_header)
                inserted_header = True
            cols = line.rstrip("\n").split("\t") + ["FORMAT", sample_name]
            new_lines.append("\t".join(cols) + "\n")
        elif line.strip():
            cols = line.rstrip("\n").split("\t") + ["GT", genotype]
            new_lines.append("\t".join(cols) + "\n")

    with open(vcf_path, "wt") as f:
        f.writelines(new_lines)

def insert_suffix_before_first_dot(path, suffix):
    dirname, basename = os.path.split(path)
    if "." in basename:
        parts = basename.split(".", 1)
        new_basename = parts[0] + suffix + "." + parts[1]
    else:
        new_basename = basename + suffix
    return os.path.join(dirname, new_basename)

def add_normalized_before_first_dot(path):
    return insert_suffix_before_first_dot(path, "_normalized")

def _vcf_has_samples(vcf_path):
    """True if the VCF carries FORMAT + per-sample genotype columns (i.e. the
    #CHROM header line has more than the 8 fixed columns)."""
    open_func = gzip.open if vcf_path.endswith(".gz") else open
    with open_func(vcf_path, "rt") as f:
        for line in f:
            if line.startswith("#CHROM"):
                return len(line.rstrip("\n").split("\t")) > 8
            if not line.startswith("#"):
                break
    return False

def filter_to_variant_calls(vcf_path):
    """Keep every record that lists a real ALT allele, regardless of genotype --
    including hom-ref (0/0) and no-call (./.) calls (e.g. bcftools/DeepVariant sites
    that still carry an alternate allele). Only gVCF reference blocks (ALT=".") are
    dropped. The FILTER column is
    preserved so hap.py can still stratify ALL vs PASS."""
    # Sites-only VCFs (no FORMAT/sample columns, e.g. varseek or VarTrix) carry no
    # genotypes, so an ALT-based filter still applies; but there is nothing to drop
    # -- every record is already a variant site -- so pass the
    # VCF through unchanged (and skip the _nonref suffix downstream).
    if not _vcf_has_samples(vcf_path):
        return vcf_path

    out_path = insert_suffix_before_first_dot(vcf_path, "_nonref")

    if os.path.isfile(out_path) and os.path.getsize(out_path) > 0:
        print(f"Filtered file {out_path} already exists. Skipping filtering.")
    else:
        output_type = "-Oz" if out_path.endswith(".vcf.gz") else "-Ov"
        subprocess.run(
            ["bcftools", "view", "-i", 'ALT!="."', output_type, "-o", out_path, vcf_path],
            check=True,
        )
    return out_path

def make_normalized_vcf(test_vcf, reference_fasta):
    test_vcf_unnormalized = test_vcf
    test_vcf = add_normalized_before_first_dot(test_vcf_unnormalized)

    # Treat a zero-byte file as missing: a previous bcftools failure can leave an
    # empty output behind, and a plain isfile() check would silently reuse it.
    if os.path.isfile(test_vcf) and os.path.getsize(test_vcf) > 0:
        print(f"Normalized file {test_vcf} already exists. Skipping normalization.")
    else:
        reference_fasta_index = f"{reference_fasta}.fai"
        if not os.path.isfile(reference_fasta_index):
            subprocess.run(["samtools", "faidx", reference_fasta], check=True)

        output_type = "-Oz" if test_vcf.endswith(".vcf.gz") else "-Ov"
        # Pipeline: reheader -> norm -> sort.
        #  * reheader --fai: the varseek VCF carries no ##contig header lines, which
        #    makes `bcftools norm` abort with "CONTIG id not present in the header"
        #    and leaves an empty file. Add/refresh contig lines from the reference.
        #  * norm: left-align, split multiallelics, check against the reference.
        #  * sort: the varseek records are not grouped by chromosome, so tabix
        #    indexing later fails with "chromosome blocks not continuous".
        reheader_proc = subprocess.Popen(
            ["bcftools", "reheader", "--fai", reference_fasta_index, test_vcf_unnormalized],
            stdout=subprocess.PIPE,
        )
        norm_proc = subprocess.Popen(
            ["bcftools", "norm", "-c", "w", "-f", reference_fasta, "-m", "-both", "-Ou"],
            stdin=reheader_proc.stdout, stdout=subprocess.PIPE,
        )
        reheader_proc.stdout.close()
        subprocess.run(["bcftools", "sort", output_type, "-o", test_vcf], stdin=norm_proc.stdout, check=True)
        norm_proc.stdout.close()
        if reheader_proc.wait() != 0:
            raise subprocess.CalledProcessError(reheader_proc.returncode, "bcftools reheader")
        if norm_proc.wait() != 0:
            raise subprocess.CalledProcessError(norm_proc.returncode, "bcftools norm")

    # sites-only VCFs (e.g. varseek) need a placeholder genotype for hap.py
    ensure_genotype_column(test_vcf)

    if test_vcf.endswith(".gz") and not os.path.isfile(f"{test_vcf}.tbi"):
        subprocess.run(["bcftools", "index", "-f", "-t", test_vcf], check=True)

    return test_vcf

def compare_with_happy(varseek_vcf, alternate_vcf, reference_fasta, output_dir, output_prefix="happy"):
    varseek_vcf = os.path.abspath(varseek_vcf)
    alternate_vcf = os.path.abspath(alternate_vcf)
    reference_fasta = os.path.abspath(reference_fasta)
    output_dir = os.path.abspath(output_dir)
    
    varseek_vcf_dir = os.path.dirname(varseek_vcf)
    alternate_vcf_dir = os.path.dirname(alternate_vcf)
    reference_fasta_dir = os.path.dirname(reference_fasta)

    reference_fasta_index = f"{reference_fasta}.fai"
    if not os.path.isfile(reference_fasta_index):
        subprocess.run(["samtools", "faidx", reference_fasta], check=True)

    # Drop non-variant (hom-ref / no-call) records from the caller VCF before
    # comparison; varseek emits only variant sites so it needs no such filter.
    alternate_vcf = filter_to_variant_calls(alternate_vcf)

    if not is_vcf_normalized(alternate_vcf):
        alternate_vcf = make_normalized_vcf(alternate_vcf, reference_fasta)

    if not is_vcf_normalized(varseek_vcf):
        varseek_vcf = make_normalized_vcf(varseek_vcf, reference_fasta)

    summary_csv_path = os.path.join(output_dir, f"{output_prefix}.summary.csv")

    if os.path.isfile(summary_csv_path):
        print(f"Summary file {summary_csv_path} already exists. Skipping hap.py run.")
    else:
        os.makedirs(output_dir, exist_ok=True)
        output_prefix_full = os.path.join(output_dir, output_prefix)
        command = f"podman run --rm -v {varseek_vcf_dir}:{varseek_vcf_dir} -v {alternate_vcf_dir}:{alternate_vcf_dir} -v {reference_fasta_dir}:{reference_fasta_dir} -v {output_dir}:{output_dir} mgibio/hap.py:v0.3.12 /opt/hap.py/bin/hap.py -r {reference_fasta} --engine=scmp-somatic -o {output_prefix_full} {varseek_vcf} {alternate_vcf}"
        subprocess.run(command, shell=True, check=True)


time: 4.07 ms (started: 2026-06-15 08:45:35 -07:00)


## COSMIC comparison

In [7]:
# vk ref out
vk_ref_out_dir = os.path.join("data", "varseek_ref_out_pbmc_1k_v3_cosmic")
vcrs_index = os.path.join(vk_ref_out_dir, "vcrs_index_denovo.idx")
vcrs_t2g = os.path.join(vk_ref_out_dir, "vcrs_t2g_denovo.txt")

# vk count out
vk_count_out_dir = os.path.join("data", "varseek_count_out_pbmc_1k_v3_cosmic")
vk_count_variants_vcf_cosmic = os.path.join(vk_count_out_dir, "varseek_variants.vcf")

time: 412 μs (started: 2026-06-15 08:45:35 -07:00)


In [8]:
cosmic_tsv = "data/cosmic/Cosmic_MutantCensus_v104_GRCh38.tsv"

if not os.path.exists(cosmic_tsv):
    raise FileNotFoundError(f"Cosmic TSV not found at {cosmic_tsv}; please download from https://cancer.sanger.ac.uk/cosmic/download/cosmic/v104/mutantcensus.")

cosmic_tsv_cols = pd.read_csv(cosmic_tsv, sep="\t", nrows=0).columns
if "seq_id" not in cosmic_tsv_cols or "variant" not in cosmic_tsv_cols:
    cosmic_df = pd.read_csv(cosmic_tsv, sep="\t")
    cosmic_df[["seq_id", "variant"]] = cosmic_df["HGVSG"].str.split(":", n=1, expand=True)
    cosmic_df.to_csv(cosmic_tsv, sep="\t", index=False)

time: 8.34 ms (started: 2026-06-15 08:45:35 -07:00)


In [9]:
if not os.path.exists(vcrs_index):
    vk_ref_output_dict = vk.ref(
        variants=cosmic_tsv,
        sequences=sequences,
        seq_id_column="seq_id",
        var_column="variant",
        out=vk_ref_out_dir,
        reference_out_dir=reference_dir,
        dlist_reference_source="t2t",
        index_out=vcrs_index,
        t2g_out=vcrs_t2g,
        w=w,
        k=k
    )

08:45:35 - INFO - Using COSMIC email from COSMIC_EMAIL environment variable


08:45:35 - INFO - Using COSMIC password from COSMIC_PASSWORD environment variable


08:45:35 - INFO - Running vk build


08:45:35 - INFO - Using COSMIC email from COSMIC_EMAIL environment variable: jmrich@caltech.edu


08:45:35 - INFO - Using COSMIC password from COSMIC_PASSWORD environment variable


...

06:09:54 - INFO - Running kb ref with command: kb ref --workflow custom -t 2 -i data/varseek_ref_out_pbmc_1k_v3_cosmic/vcrs_index_denovo.idx --d-list None -k 41 --overwrite data/varseek_ref_out_pbmc_1k_v3_cosmic/vcrs_filtered.fa


[2026-06-16 06:14:47,599] WARNING [ref_custom] Using provided k-mer length 41 instead of optimal length 31
[2026-06-16 06:14:47,599]    INFO [ref_custom] Indexing data/varseek_ref_out_pbmc_1k_v3_cosmic/vcrs_filtered.fa to data/varseek_ref_out_pbmc_1k_v3_cosmic/vcrs_index_denovo.idx


[2026-06-16 06:15:59,447]    INFO [ref_custom] Finished creating custom index
06:15:59 - INFO - Produced files: {'index': '/home/jrich/Desktop/varseek-examples/data/varseek_ref_out_pbmc_1k_v3_cosmic/vcrs_index_denovo.idx', 't2g': '/home/jrich/Desktop/varseek-examples/data/varseek_ref_out_pbmc_1k_v3_cosmic/vcrs_t2g_denovo.txt', 'fasta': '/home/jrich/Desktop/varseek-examples/data/varseek_ref_out_pbmc_1k_v3_cosmic/vcrs_filtered.fa'}


06:15:59 - INFO - Total runtime for vk ref: 1290m, 24.02s


time: 21h 30min 24s (started: 2026-06-15 08:45:35 -07:00)


In [10]:
vk_count_output_dict = vk.count(
    fastqs_dir,
    index=vcrs_index,
    t2g=vcrs_t2g,
    technology=technology,
    out=vk_count_out_dir,
    k=k,
    threads=threads,
    strand="unstranded",
    min_counts=min_counts_clean,
)

06:16:01 - INFO - Removing index files from fastq files list, as they are not utilized in kb count with technology 10XV3


06:16:01 - INFO - File data/pbmc_1k_v3_fastqs/combined_R2.fastq.gz does not match the Illumina file naming convention of SAMPLE_LANE_R[12]_001.fastq.gz, so removing it from the fastq files list.


06:16:01 - INFO - Setting length_required to 41 if fastqpp is run


06:16:01 - INFO - Skipping vk fastqpp because there was no use for it


06:16:01 - INFO - Running kb count with command: kb count -t 16 -k 41 -i data/varseek_ref_out_pbmc_1k_v3_cosmic/vcrs_index_denovo.idx -g data/varseek_ref_out_pbmc_1k_v3_cosmic/vcrs_t2g_denovo.txt -x 10XV3 --h5ad -o data/varseek_count_out_pbmc_1k_v3_cosmic/kb_count_out_vcrs --overwrite --strand unstranded --num --mm --union data/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L001_R1_001.fastq.gz data/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L001_R2_001.fastq.gz data/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L002_R1_001.fastq.gz data/pbmc_1k_v3_fastqs/pbmc_1k_v3_S1_L002_R2_001.fastq.gz


...

06:22:26 - INFO - Saving adata to data/varseek_count_out_pbmc_1k_v3_cosmic/adata_cleaned.h5ad.


06:23:12 - INFO - Total runtime for vk clean: 1m, 44.21s


06:23:12 - INFO - Skipping vk summarize because summarize=False


06:23:12 - INFO - Total runtime for vk count: 7m, 13.16s


time: 7min 13s (started: 2026-06-16 06:15:59 -07:00)


In [11]:
adata_path = vk_count_output_dict['adata_path']  #  if not use_unfiltered_varseek else vk_count_output_dict['adata_path_unprocessed']
adata_to_vcf(adata_path, vk_count_variants_vcf_cosmic)

Could not parse: 7:g.28158166_28158167dup
Could not parse: 7:g.28134727dup
Could not parse: 7:g.27988697dup
Could not parse: 17:g.17212311dup
Could not parse: 3:g.185392894dup
Could not parse: 9:g.95203240dup
Could not parse: 4:g.53453093_53453094dup
Could not parse: 4:g.53453030_53453031dup
Could not parse: X:g.119633440dup
Could not parse: 11:g.3737477dup
Could not parse: 11:g.3722476_3722477dup
Could not parse: 11:g.3697661dup
Could not parse: 6:g.117561913dup
Could not parse: 6:g.167969855dup
Could not parse: 11:g.72048552dup
Could not parse: 13:g.48307443dup
Could not parse: 13:g.48411476_48411478inv
Could not parse: 9:g.136539388dup
Could not parse: 2:g.113249901dup
Could not parse: 19:g.10793867dup
Could not parse: 19:g.54111988dup
Could not parse: 7:g.20140901_20140902dup
Could not parse: 7:g.20140901_20140902dup
Could not parse: 6:g.29942780_29942781delinsAC
Could not parse: 2:g.99868292dup
Could not parse: 2:g.99580905dup
Could not parse: 2:g.99874184dup
Could not parse: 2:g.

Wrote 6,270 variants to data/varseek_count_out_pbmc_1k_v3_cosmic/varseek_variants.vcf
time: 11.2 s (started: 2026-06-16 06:23:12 -07:00)


In [12]:
happy_out = os.path.join("data", "happy_output_cosmic")
label = "varseek_cosmic"
# if not use_unfiltered_varseek:
#     happy_out += "_filtered"

compare_with_happy(
    varseek_vcf=vk_count_variants_vcf,
    alternate_vcf=vk_count_variants_vcf_cosmic,
    reference_fasta=sequences,
    output_dir=happy_out,
    output_prefix=f"varseek_vs_{label}"
)

Writing to /home/jrich/tmp/bcftools.1wDELG


Lines   total/split/joined/realigned/removed/skipped:	6270/0/0/673/0/0
Merging 1 temporary files
Done
Cleaning


Normalized file /home/jrich/Desktop/varseek-examples/data/varseek_count_out_pbmc_1k_v3/varseek_variants_normalized.vcf already exists. Skipping normalization.


2026-06-16 13:23:39,007 WARNING  No reference file found at default locations. You can set the environment variable 'HGREF' or 'HG19' to point to a suitable Fasta file.


[W] overlapping records at 12:66227605 for sample 0


[W] Variants that overlap on the reference allele: 10
[I] Total VCF records:         230680
[I] Non-reference VCF records: 230680


[W] overlapping records at 1:204557929 for sample 0
[W] Variants that overlap on the reference allele: 24
[I] Total VCF records:         6270
[I] Non-reference VCF records: 6270
2026-06-16 13:23:45,187 WARNING  No calls for location MT in query!
2026-06-16 13:23:45,187 WARNING  No calls for location Y in query!


Hap.py v0.3.12


Benchmarking Summary:
Type Filter  TRUTH.TOTAL  TRUTH.TP  TRUTH.FN  QUERY.TOTAL  QUERY.FP  QUERY.UNK  FP.gt  FP.al  METRIC.Recall  METRIC.Precision  METRIC.Frac_NA  METRIC.F1_Score  TRUTH.TOTAL.TiTv_ratio  QUERY.TOTAL.TiTv_ratio  TRUTH.TOTAL.het_hom_ratio  QUERY.TOTAL.het_hom_ratio
INDEL    ALL        16331        42     16289          714       671          0      0      0       0.002572          0.060224             0.0         0.004933                     NaN                     NaN                        NaN                        NaN
INDEL   PASS        16331        42     16289          714       671          0      0      0       0.002572          0.060224             0.0         0.004933                     NaN                     NaN                        NaN                        NaN
  SNP    ALL       214349       428    213921         5556      5129          0      0      0       0.001997          0.076854             0.0         0.003892                3.681233          

time: 58.5 s (started: 2026-06-16 06:23:24 -07:00)


In [13]:
# Collect the hap.py summary tables into one view.
# varseek is the TRUTH set and each caller is the QUERY, so:
#   Recall    = fraction of varseek calls confirmed by the caller
#   Precision = fraction of the caller's calls also in varseek (low by design:
#               varseek reports a small targeted subset vs. a genome-wide caller)
summary_frames = []
for label, alternate_vcf in [("varseek_cosmic", vk_count_variants_vcf_cosmic)]:
    summary_csv_path = os.path.join(happy_out, f"varseek_vs_{label}.summary.csv")
    if not os.path.isfile(summary_csv_path):
        continue
    df = pd.read_csv(summary_csv_path)
    df.insert(0, "Caller", label)
    summary_frames.append(df)

if summary_frames:
    summary_all = pd.concat(summary_frames, ignore_index=True)
    cols = ["Caller", "Type", "Filter", "TRUTH.TOTAL", "TRUTH.TP", "TRUTH.FN",
            "QUERY.TOTAL", "QUERY.FP", "METRIC.Recall", "METRIC.Precision", "METRIC.F1_Score"]
    display(summary_all[cols])
else:
    print("No hap.py summaries found yet -- run the comparison cell above first.")


,Caller,Type,Filter,TRUTH.TOTAL,TRUTH.TP,TRUTH.FN,QUERY.TOTAL,QUERY.FP,METRIC.Recall,METRIC.Precision,METRIC.F1_Score
0,varseek_cosmic,INDEL,ALL,16331,42,16289,714,671,0.002572,0.060224,0.004933
1,varseek_cosmic,INDEL,PASS,16331,42,16289,714,671,0.002572,0.060224,0.004933
2,varseek_cosmic,SNP,ALL,214349,428,213921,5556,5129,0.001997,0.076854,0.003892
3,varseek_cosmic,SNP,PASS,214349,428,213921,5556,5129,0.001997,0.076854,0.003892


time: 34.1 ms (started: 2026-06-16 06:24:22 -07:00)
